# ECG Classification System â€” Master Thesis Prototype
**Mohammed Irshad Kunnam Puthoor | VMU | MSc Applied Informatics**

### Setup Instructions
1. On Kaggle: Add the PTB-XL dataset â†’ Search 'PTB-XL' in Add Data â†’ select `ptb-xl-a-large-publicly-available-electrocardiography-dataset`
2. Set Runtime â†’ GPU (T4 x2 recommended)
3. Run all cells

In [ ]:
# Install dependencies
!pip install wfdb xgboost psutil -q

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import time
import numpy as np
import pandas as pd
import wfdb
import psutil
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import tensorflow as tf
from scipy.signal import butter, filtfilt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, roc_curve, auc
)
from tensorflow.keras import layers, Model, Input, optimizers
from xgboost import XGBClassifier

print('All imports OK')
print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# â”€â”€ CONFIG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Kaggle PTB-XL dataset path
DATA_DIR = '/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1'
CSV_PATH = os.path.join(DATA_DIR, 'ptbxl_database.csv')
SCP_PATH = os.path.join(DATA_DIR, 'scp_statements.csv')

# Data
TOTAL_RECORDS_TO_LOAD = 21430   # Full PTB-XL dataset
MAX_SAMPLES           = 1000    # 10 seconds at 100Hz
CHANNELS              = 12      # Full 12-lead ECG

# Model
LATENT_DIM  = 128
PATCH_SIZE  = 50
MASK_RATIO  = 0.75
EPOCHS_AE   = 30               # More epochs on GPU
EPOCHS_DL   = 30
BATCH_SIZE  = 64               # Larger batch for GPU
TEST_SIZE   = 0.2
RANDOM_STATE = 42

LABEL_MAP = {'NORM': 0, 'MI': 1, 'STTC': 2, 'HYP': 3, 'CD': 4}

def log(msg):
    print(f'[INFO] {msg}')

log('Config loaded')

In [ ]:
# â”€â”€ DATA LOADER â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def apply_filter(data, lowcut=0.5, highcut=40.0, fs=100.0, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, data, axis=0)

def load_and_process():
    log('Loading PTB-XL metadata...')
    df = pd.read_csv(CSV_PATH, index_col='ecg_id')
    agg_df = pd.read_csv(SCP_PATH, index_col=0)

    diag_col = None
    for col in ['diagnostic_class', 'diagnostic_superclass', 'diagnostic']:
        if col in agg_df.columns:
            diag_col = col
            break
    log(f'Using diagnostic column: {diag_col}')
    agg_df = agg_df[agg_df[diag_col].notnull()]

    def aggregate_diagnostic(y_dic):
        if isinstance(y_dic, str):
            y_dic = eval(y_dic)
        tmp = [agg_df.loc[k][diag_col] for k in y_dic if k in agg_df.index]
        tmp = list(set(tmp))
        return tmp[0] if tmp else None

    df['label'] = df.scp_codes.apply(aggregate_diagnostic)
    df = df.dropna(subset=['label'])
    df = df[df['label'].isin(LABEL_MAP.keys())]
    df['label_id'] = df['label'].map(LABEL_MAP)
    log(f'Total valid records: {len(df)}. Sampling {TOTAL_RECORDS_TO_LOAD}...')

    df = df.sample(n=min(TOTAL_RECORDS_TO_LOAD, len(df)), random_state=RANDOM_STATE)

    n = len(df)
    X = np.zeros((n, MAX_SAMPLES, CHANNELS), dtype=np.float32)
    y = np.zeros(n, dtype=np.int32)
    count = 0

    for _, row in df.iterrows():
        try:
            signal, _ = wfdb.rdsamp(os.path.join(DATA_DIR, row['filename_lr']))
            if signal.shape[1] < 12:
                continue
            ecg = signal[:MAX_SAMPLES, :]
            if len(ecg) == MAX_SAMPLES:
                X[count] = apply_filter(ecg).astype(np.float32)
                y[count] = row['label_id']
                count += 1
        except Exception:
            continue

    X, y = X[:count], y[:count]
    log(f'Loaded {count} records. Shape: {X.shape}')

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train.reshape(-1, CHANNELS)).reshape(X_train.shape)
    X_test  = scaler.transform(X_test.reshape(-1, CHANNELS)).reshape(X_test.shape)

    return X_train.astype(np.float32), X_test.astype(np.float32), y_train, y_test

X_train, X_test, y_train, y_test = load_and_process()
log(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# â”€â”€ MAE MASKING â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def apply_mae_mask(X):
    X_masked = X.copy()
    num_patches = X.shape[1] // PATCH_SIZE
    num_mask = int(num_patches * MASK_RATIO)
    for i in range(X.shape[0]):
        for idx in np.random.choice(num_patches, num_mask, replace=False):
            X_masked[i, idx*PATCH_SIZE:(idx+1)*PATCH_SIZE, :] = 0
    return X_masked

log('Generating MAE masked data...')
X_train_masked = apply_mae_mask(X_train)
log('Done')

In [ ]:
# â”€â”€ MODELS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def build_autoencoder(input_shape=(MAX_SAMPLES, CHANNELS)):
    enc_in = Input(shape=input_shape)
    x = layers.Conv1D(32, 5, activation='relu', padding='same')(enc_in)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Flatten()(x)
    latent = layers.Dense(LATENT_DIM, activation='relu', name='latent_features')(x)
    encoder = Model(enc_in, latent, name='Encoder')

    dec_in = Input(shape=(LATENT_DIM,))
    x = layers.Dense((input_shape[0]//4) * 64, activation='relu')(dec_in)
    x = layers.Reshape((input_shape[0]//4, 64))(x)
    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(32, 5, activation='relu', padding='same')(x)
    x = layers.UpSampling1D(2)(x)
    out = layers.Conv1D(CHANNELS, 5, activation='linear', padding='same')(x)
    decoder = Model(dec_in, out, name='Decoder')

    ae_out = decoder(encoder(enc_in))
    ae = Model(enc_in, ae_out, name='Autoencoder')
    ae.compile(optimizer=optimizers.Adam(0.001), loss='mse')
    return ae, encoder

def build_bilstm(num_classes=5):
    inp = Input(shape=(MAX_SAMPLES, CHANNELS))
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inp)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    m = Model(inp, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def build_cnn(num_classes=5):
    inp = Input(shape=(MAX_SAMPLES, CHANNELS))
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inp)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    m = Model(inp, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

log('Model builders ready')

In [ ]:
# â”€â”€ TRAIN AUTOENCODERS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
log('Training SAE...')
sae, encoder_sae = build_autoencoder()
sae.fit(X_train, X_train, epochs=EPOCHS_AE, batch_size=BATCH_SIZE, verbose=1)

log('Training MAE (Self-Supervised)...')
mae, encoder_mae = build_autoencoder()
mae.fit(X_train_masked, X_train, epochs=EPOCHS_AE, batch_size=BATCH_SIZE, verbose=1)

In [ ]:
# â”€â”€ EXTRACT FEATURES â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
log('Extracting latent features...')
X_train_sae = encoder_sae.predict(X_train, verbose=0)
X_test_sae  = encoder_sae.predict(X_test,  verbose=0)
X_train_mae = encoder_mae.predict(X_train, verbose=0)
X_test_mae  = encoder_mae.predict(X_test,  verbose=0)

log('Fitting PCA baseline...')
pca = PCA(n_components=LATENT_DIM)
X_train_pca = pca.fit_transform(X_train.reshape(X_train.shape[0], -1))
X_test_pca  = pca.transform(X_test.reshape(X_test.shape[0], -1))
log('Feature extraction complete')

In [ ]:
# â”€â”€ TRAIN CLASSIFIERS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
research_data = {}
latency_data  = {}
proba_data    = {}

ml_pipelines = [
    ('PCA + SVM',        X_train_pca, X_test_pca, SVC(probability=True, max_iter=3000)),
    ('SAE + SVM',        X_train_sae, X_test_sae, SVC(probability=True, max_iter=3000)),
    ('MAE + SVM (Ours)', X_train_mae, X_test_mae, SVC(probability=True, max_iter=3000)),
    ('MAE + XGBoost',    X_train_mae, X_test_mae, XGBClassifier(eval_metric='mlogloss', n_jobs=-1)),
]

for name, x_tr, x_te, clf in ml_pipelines:
    log(f'Training {name}...')
    clf.fit(x_tr, y_train)
    t0 = time.time()
    preds = clf.predict(x_te)
    elapsed = time.time() - t0
    proba = clf.predict_proba(x_te)
    research_data[name] = {'accuracy': accuracy_score(y_test, preds), 'y_pred': preds}
    latency_data[name]  = elapsed
    proba_data[name]    = proba
    log(f'  Accuracy: {accuracy_score(y_test, preds):.3f} | Latency: {elapsed*1000:.1f}ms')

log('Training BiLSTM...')
bilstm = build_bilstm()
bilstm.fit(X_train, y_train, epochs=EPOCHS_DL, batch_size=BATCH_SIZE, verbose=1)
t0 = time.time()
bilstm_probs = bilstm.predict(X_test, verbose=0)
elapsed = time.time() - t0
bilstm_preds = np.argmax(bilstm_probs, axis=1)
research_data['Raw + BiLSTM'] = {'accuracy': accuracy_score(y_test, bilstm_preds), 'y_pred': bilstm_preds}
latency_data['Raw + BiLSTM'] = elapsed
proba_data['Raw + BiLSTM']   = bilstm_probs
log(f'  BiLSTM Accuracy: {accuracy_score(y_test, bilstm_preds):.3f}')

log('Training 1D-CNN...')
cnn = build_cnn()
cnn.fit(X_train, y_train, epochs=EPOCHS_DL, batch_size=BATCH_SIZE, verbose=1)
t0 = time.time()
cnn_probs = cnn.predict(X_test, verbose=0)
elapsed = time.time() - t0
cnn_preds = np.argmax(cnn_probs, axis=1)
research_data['Raw + 1D-CNN'] = {'accuracy': accuracy_score(y_test, cnn_preds), 'y_pred': cnn_preds}
latency_data['Raw + 1D-CNN'] = elapsed
proba_data['Raw + 1D-CNN']   = cnn_probs
log(f'  1D-CNN Accuracy: {accuracy_score(y_test, cnn_preds):.3f}')

In [ ]:
# â”€â”€ VISUALIZATIONS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CLASS_NAMES = list(LABEL_MAP.keys())
sns.set_theme(style='whitegrid')

# 1. MAE Masking Strategy
_, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(X_train[0, :, 0], color='steelblue', linewidth=0.8)
ax1.set_title('Original ECG Signal (Lead I)', fontweight='bold')
ax1.set_ylabel('Amplitude')
ax2.plot(X_train_masked[0, :, 0], color='crimson', linewidth=0.8)
ax2.set_title(f'Masked Input (75% hidden)', fontweight='bold')
ax2.set_ylabel('Amplitude')
ax2.set_xlabel('Samples')
plt.suptitle('MAE Self-Supervised Training Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_mae_masking.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_mae_masking.png')

In [ ]:
# 2. MAE Reconstruction Quality
sample = X_train_masked[0:1]
reconstructed = mae.predict(sample, verbose=0)[0]
_, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(X_train[0, :, 0], color='steelblue', linewidth=0.8)
axes[0].set_title('1. Original ECG (Ground Truth)', fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[1].plot(X_train_masked[0, :, 0], color='crimson', linewidth=0.8)
axes[1].set_title('2. Masked Input (75% hidden)', fontweight='bold')
axes[1].set_ylabel('Amplitude')
axes[2].plot(reconstructed[:, 0], color='forestgreen', linewidth=0.8)
axes[2].set_title('3. MAE Reconstruction', fontweight='bold')
axes[2].set_ylabel('Amplitude')
axes[2].set_xlabel('Samples')
plt.suptitle('MAE Reconstruction Quality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_reconstruction.png')

In [ ]:
# 3. Accuracy Comparison
names = list(research_data.keys())
accs  = [research_data[n]['accuracy'] for n in names]
colors = ['#e63946' if 'Ours' in n else '#2d004b' for n in names]
plt.figure(figsize=(12, 6))
bars = plt.bar(names, accs, color=colors, edgecolor='white')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
             f'{acc:.1%}', ha='center', fontweight='bold', fontsize=11)
plt.title('ECG Classification Accuracy: All Models', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('outputs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_accuracy.png')

In [ ]:
# 4. F1 Heatmap
f1_matrix = []
for _, data in research_data.items():
    f1 = f1_score(y_test, data['y_pred'], average=None,
                  labels=list(range(5)), zero_division=0)
    f1_matrix.append(f1)
plt.figure(figsize=(10, 6))
sns.heatmap(np.array(f1_matrix), annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=names,
            vmin=0, vmax=1, linewidths=0.5, annot_kws={'size': 11, 'weight': 'bold'})
plt.title('Per-Class F1 Score', fontsize=13, fontweight='bold')
plt.xlabel('Cardiac Condition')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('outputs_f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_f1_heatmap.png')

In [ ]:
# 5. Inference Latency
times = [latency_data[n]*1000 for n in names]
colors2 = ['#e63946' if 'MAE' in n else '#457b9d' for n in names]
_, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(names, times, color=colors2, edgecolor='white')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{t:.1f}ms', ha='center', fontweight='bold', fontsize=10)
ax.set_title('Inference Latency per Model', fontsize=13, fontweight='bold')
ax.set_ylabel('Time (ms)')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=15, ha='right')
plt.tight_layout()
plt.savefig('outputs_latency.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_latency.png')

In [ ]:
# 6. t-SNE Latent Space
_, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors3 = ['#e63946','#457b9d','#2a9d8f','#e9c46a','#f4a261']
for features, ax, title in [(X_test_sae, ax1, 'SAE'), (X_test_mae, ax2, 'MAE (Ours)')]:
    reduced = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30, max_iter=500).fit_transform(features)
    for i, cls in enumerate(CLASS_NAMES):
        mask = y_test == i
        ax.scatter(reduced[mask,0], reduced[mask,1], c=colors3[i], label=cls, alpha=0.6, s=20)
    ax.set_title(f'{title} Latent Space', fontweight='bold')
    ax.legend(markerscale=2)
plt.suptitle('Latent Space t-SNE Visualization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_tsne.png')

In [ ]:
# 7. XAI Saliency Map
sample_idx = 0
x_tensor = tf.convert_to_tensor(X_test[sample_idx:sample_idx+1], dtype=tf.float32)
with tf.GradientTape() as tape:
    tape.watch(x_tensor)
    pred = cnn(x_tensor, training=False)
    predicted_class = int(tf.argmax(pred[0]))
    score = pred[0, predicted_class]
grads = tape.gradient(score, x_tensor)[0].numpy()
saliency = np.mean(np.abs(grads), axis=1)
saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
_, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(X_test[sample_idx, :, 0], color='steelblue', linewidth=0.8)
ax1.set_title(f'XAI Saliency â€” 1D-CNN | True: {CLASS_NAMES[y_test[sample_idx]]} | Predicted: {CLASS_NAMES[predicted_class]}',
              fontweight='bold')
ax1.set_ylabel('Amplitude')
ax2.imshow(saliency[np.newaxis,:], aspect='auto', cmap='hot', extent=[0, len(saliency), 0, 1])
ax2.set_title('Saliency Intensity (red = high influence)', fontsize=9)
ax2.set_xlabel('Samples')
plt.suptitle('Explainability (XAI)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_saliency.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_saliency.png')

In [ ]:
# 8. Precision & Recall Heatmaps
p_matrix = [precision_score(y_test, d['y_pred'], average=None, labels=list(range(5)), zero_division=0) for d in research_data.values()]
r_matrix = [recall_score(y_test, d['y_pred'], average=None, labels=list(range(5)), zero_division=0) for d in research_data.values()]
_, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(np.array(p_matrix), annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=names, vmin=0, vmax=1, ax=ax1,
            annot_kws={'size': 11, 'weight': 'bold'})
ax1.set_title('Precision per Class', fontweight='bold')
sns.heatmap(np.array(r_matrix), annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=names, vmin=0, vmax=1, ax=ax2,
            annot_kws={'size': 11, 'weight': 'bold'})
ax2.set_title('Recall per Class', fontweight='bold')
plt.suptitle('Precision & Recall', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_precision_recall.png')

In [ ]:
# 9. AUC-ROC Curves
y_bin = label_binarize(y_test, classes=list(range(5)))
n_models = len(proba_data)
_, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, (name, proba) in enumerate(proba_data.items()):
    ax = axes[i]
    for j, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, j], proba[:, j])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=1.5, label=f'{cls} (AUC={roc_auc:.2f})')
    ax.plot([0,1],[0,1],'k--', alpha=0.4)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=8)
for j in range(n_models, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('AUC-ROC Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_roc.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_roc.png')

In [ ]:
# 10. Confusion Matrices
_, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, (name, data) in enumerate(research_data.items()):
    cm = confusion_matrix(y_test, data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[i].set_title(f"{name}\nAcc: {data['accuracy']:.1%}", fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
for j in range(len(research_data), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Confusion Matrices: All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
log('Saved: outputs_confusion.png')

In [ ]:
# â”€â”€ FINAL SUMMARY â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
for name, data in research_data.items():
    f1 = f1_score(y_test, data['y_pred'], average='macro', zero_division=0)
    lat = latency_data[name]*1000
    print(f"{name:<22} Acc: {data['accuracy']:.3f}  Macro-F1: {f1:.3f}  Latency: {lat:.1f}ms")
print('='*60)
print('All output images saved!')